In [380]:
# !pip install pandas pyjanitor thefuzz --quiet

In [381]:
import pandas as pd
import numpy as np
import janitor
from thefuzz import process, fuzz

In [382]:
orders = pd.read_csv('messy_orders.csv')
customers = pd.read_csv('customer_master.csv')

In [383]:
orders.head()

,Order ID,Customer ID,Customer Details,Product Category,Unit Price,Quantity,Order Date,Sales Channel,Order Total
0,ORD-000001,C00347,Leo Lewis | Germany | Basic,HOME & KITCHEN,348.61,1,2026-06-10,Website,348.61
1,ORD-000002,C00966,Nora Baker | Australia | Plus,Clothng,274.73,5,2025-02-21,Website,1373.65
2,ORD-000003,C01862,Lina Clark | United Kingdom | Basic,ELECTRONICS,194.69,4,2025-02-06,Mobile App,778.76
3,ORD-000004,C02451,Amelia Hall | France | Basic,Beauty,631.77,5,2025-09-15,Marketplace,3158.85
4,ORD-000005,C00730,Adam Walker | Singapore | Plus,Clothing & Apparel,324.89,1,2025-06-26,Mobile App,324.89


In [384]:
# Get unique categories from the dataset

categories = orders['Product Category'].dropna().unique().tolist()

len(categories)

41

In [385]:
print(sorted(categories))

['BEAUTY', 'BOOKS', 'Beauty', 'Beauty Products', 'Beautyy', 'Boks', 'Book', 'Books', 'Books & Media', 'CLOTHING', 'Clothes', 'Clothing', 'Clothing & Apparel', 'Clothng', 'ELECTRONICS', 'Electroncs', 'Electronic Goods', 'Electronics', 'Electronics ', 'HOME & KITCHEN', 'Home & Kitchen', 'Home & Ktchen', 'Home Kitchen', 'Home and Kitchen', 'OFFICE SUPPLIES', 'Office  Supplies', 'Office Supples', 'Office Supplies', 'Office Supply', 'SPORTS', 'Sport', 'Sports', 'Sports & Fitness', 'Sprots', 'TOYS', 'Toy', 'Toys', 'Toys ', 'Toys & Games', 'beauty ', 'electronic']


In [386]:
# To check % of similarity in algorithms

for cat in['BOOKS', 'CLOTHING', 'ELECTRONICS']:
    print(process.extract(cat, categories, scorer=fuzz.WRatio, limit=8))

[('Books', 100), ('BOOKS', 100), ('Books & Media', 90), ('Book', 89), ('Boks', 89), ('Electronic Goods', 60), ('Toys', 44), ('Toys ', 44)]
[('Clothing', 100), ('CLOTHING', 100), ('Clothng', 93), ('Clothing & Apparel', 90), ('Clothes', 67), ('HOME & KITCHEN', 45), ('Home & Kitchen', 45), ('Home and Kitchen', 45)]
[('ELECTRONICS', 100), ('Electronics', 100), ('Electronics ', 100), ('Electroncs', 95), ('electronic', 95), ('Electronic Goods', 81), ('Toy', 60), ('Beauty Products', 46)]


In [387]:
# Define the correct category names

correct_categories = [
    'Electronics',
    'Home & Kitchen',
    'Office Supplies',
    'Clothing',
    'Beauty',
    'Books',
    'Sports',
    'Toys'
]

In [388]:
# Fuzzy matching

mapping = {}
match_results = []

for category in categories:
    match, score = process.extractOne(
        category,
        correct_categories,
        scorer=fuzz.WRatio
    )

    match_results.append({
        'original_category': category,
        'suggested_category': match,
        'similarity_score': score
    })

    if score >= 85:
        mapping[category] = match

In [389]:
# Review fuzzy matches

match_review = pd.DataFrame(match_results)

match_review.sort_values(
    'similarity_score'
)

,original_category,suggested_category,similarity_score
28,Clothes,Clothing,67
38,Electronic Goods,Electronics,81
30,Sprots,Sports,83
6,Toy,Toys,86
36,Office Supply,Office Supplies,86
14,Book,Books,89
29,Boks,Books,89
17,Books & Media,Books,90
37,Beauty Products,Beauty,90
32,Sports & Fitness,Sports,90


In [390]:
mapping

{'HOME & KITCHEN': 'Home & Kitchen',
 'Clothng': 'Clothing',
 'ELECTRONICS': 'Electronics',
 'Beauty': 'Beauty',
 'Clothing & Apparel': 'Clothing',
 'Office  Supplies': 'Office Supplies',
 'Toy': 'Toys',
 'Electroncs': 'Electronics',
 'OFFICE SUPPLIES': 'Office Supplies',
 'Clothing': 'Clothing',
 'Toys & Games': 'Toys',
 'Office Supplies': 'Office Supplies',
 'Sports': 'Sports',
 'Books': 'Books',
 'Book': 'Books',
 'Home & Kitchen': 'Home & Kitchen',
 'Toys': 'Toys',
 'Books & Media': 'Books',
 'Beautyy': 'Beauty',
 'Home and Kitchen': 'Home & Kitchen',
 'SPORTS': 'Sports',
 'Electronics': 'Electronics',
 'beauty ': 'Beauty',
 'Home & Ktchen': 'Home & Kitchen',
 'Office Supples': 'Office Supplies',
 'CLOTHING': 'Clothing',
 'Electronics ': 'Electronics',
 'electronic': 'Electronics',
 'Boks': 'Books',
 'Sport': 'Sports',
 'Sports & Fitness': 'Sports',
 'BEAUTY': 'Beauty',
 'Home Kitchen': 'Home & Kitchen',
 'Toys ': 'Toys',
 'Office Supply': 'Office Supplies',
 'Beauty Products': 'Be

In [391]:
orders['Product Category'] = (orders['Product Category'].map(mapping).fillna(orders['Product Category']))

In [392]:
orders.head()

,Order ID,Customer ID,Customer Details,Product Category,Unit Price,Quantity,Order Date,Sales Channel,Order Total
0,ORD-000001,C00347,Leo Lewis | Germany | Basic,Home & Kitchen,348.61,1,2026-06-10,Website,348.61
1,ORD-000002,C00966,Nora Baker | Australia | Plus,Clothing,274.73,5,2025-02-21,Website,1373.65
2,ORD-000003,C01862,Lina Clark | United Kingdom | Basic,Electronics,194.69,4,2025-02-06,Mobile App,778.76
3,ORD-000004,C02451,Amelia Hall | France | Basic,Beauty,631.77,5,2025-09-15,Marketplace,3158.85
4,ORD-000005,C00730,Adam Walker | Singapore | Plus,Clothing,324.89,1,2025-06-26,Mobile App,324.89


Step 2: Standardize column names

In [393]:
# Clean columns in one line with Pyjanitor

print('\nBefore:', orders.columns.tolist())
df = orders.clean_names()
print('After: ', df.columns.tolist())


Before: [' Order ID ', 'Customer ID', 'Customer Details ', 'Product Category', 'Unit Price ', 'Quantity', 'Order Date', 'Sales Channel', 'Order Total']
After:  ['order_id', 'customer_id', 'customer_details', 'product_category', 'unit_price', 'quantity', 'order_date', 'sales_channel', 'order_total']


In [394]:
df = (
    orders
    .clean_names()
    .remove_empty()
)


In [395]:
df.head()

,order_id,customer_id,customer_details,product_category,unit_price,quantity,order_date,sales_channel,order_total
0,ORD-000001,C00347,Leo Lewis | Germany | Basic,Home & Kitchen,348.61,1,2026-06-10,Website,348.61
1,ORD-000002,C00966,Nora Baker | Australia | Plus,Clothing,274.73,5,2025-02-21,Website,1373.65
2,ORD-000003,C01862,Lina Clark | United Kingdom | Basic,Electronics,194.69,4,2025-02-06,Mobile App,778.76
3,ORD-000004,C02451,Amelia Hall | France | Basic,Beauty,631.77,5,2025-09-15,Marketplace,3158.85
4,ORD-000005,C00730,Adam Walker | Singapore | Plus,Clothing,324.89,1,2025-06-26,Mobile App,324.89


Step 3: Parse dates

In [396]:
#convert date (string) to date(date)

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

print(df['order_date'].dtype)
print(customers['signup_date'].dtype)

datetime64[us]
datetime64[us]


Step 4: Parse the embedded customer details text

In [397]:
# Parse the embedded customer details

details = (
    df['customer_details']
    .str.split('|',n=2, expand=True)
    .apply(lambda col: col.str.strip())
)

details.columns = [
    'parsed_customer_name',
    'parsed_country',
    'parsed_plan'
]

df = pd.concat([df, details], axis=1)


In [398]:
details

,parsed_customer_name,parsed_country,parsed_plan
0,Leo Lewis,Germany,Basic
1,Nora Baker,Australia,Plus
2,Lina Clark,United Kingdom,Basic
3,Amelia Hall,France,Basic
4,Adam Walker,Singapore,Plus
...,...,...,...
9995,Ryan Young,Germany,Basic
9996,Ava Garcia,UAE,Basic
9997,Sophia Wilson,United Kingdom,Plus
9998,Sara Walker,Australia,Basic


In [399]:
df.head ()

,order_id,customer_id,customer_details,product_category,unit_price,quantity,order_date,sales_channel,order_total,parsed_customer_name,parsed_country,parsed_plan
0,ORD-000001,C00347,Leo Lewis | Germany | Basic,Home & Kitchen,348.61,1,2026-06-10,Website,348.61,Leo Lewis,Germany,Basic
1,ORD-000002,C00966,Nora Baker | Australia | Plus,Clothing,274.73,5,2025-02-21,Website,1373.65,Nora Baker,Australia,Plus
2,ORD-000003,C01862,Lina Clark | United Kingdom | Basic,Electronics,194.69,4,2025-02-06,Mobile App,778.76,Lina Clark,United Kingdom,Basic
3,ORD-000004,C02451,Amelia Hall | France | Basic,Beauty,631.77,5,2025-09-15,Marketplace,3158.85,Amelia Hall,France,Basic
4,ORD-000005,C00730,Adam Walker | Singapore | Plus,Clothing,324.89,1,2025-06-26,Mobile App,324.89,Adam Walker,Singapore,Plus


Step 5: Merge in the customer master, preferring verified data over parsed text

In [400]:
# Prepare the customer master lookup

customer_lookup = customers[[
    'customer_id',
    'customer_name',
    'country',
    'city',
    'plan',
    'segment',
    'signup_date'
]]

In [401]:
# Match orders to the customer master

df = df.merge(
    customer_lookup,
    on='customer_id',
    how='left',
    indicator=True,
    validate='many_to_one',
    suffixes=('', '_master')
)


In [402]:
# Prefer master data when available; otherwise retain parsed values

df['customer_name'] = df['customer_name'].fillna(df['parsed_customer_name'])
df['country'] = df['country'].fillna(df['parsed_country'])
df['plan'] = df['plan'].fillna(df['parsed_plan'])

In [403]:
# Flag customer IDs that are not present in the master table

df['customer_master_match'] = np.where(
    df['_merge'].eq('both'),
    'Matched',
    'Not Found'
)

print(df['customer_master_match'].value_counts())

customer_master_match
Matched      9475
Not Found     525
Name: count, dtype: int64


Step 6: Validate order dates against signup dates

In [404]:
# Validate order date against customer sign up date

df['order_before_signup'] = (
    df['order_date'].notna()
    & df['signup_date'].notna()
    & (df['order_date'] < df['signup_date'])
)

print('Orders before customer signup:', df['order_before_signup'].sum())

Orders before customer signup: 1545


Step 7: Negative unit prices

In [405]:
# Validate negative unit prices

df['negative_price_corrected'] = (df['unit_price'] < 0)
n_neg = df['negative_price_corrected'].sum()

In [406]:
# Correct negative unit prices

df['unit_price'] = df['unit_price'].abs()

print('Negative unit price corrected:', n_neg)
print('Negative unit prices remaining:', (df['unit_price'] < 0).sum())

Negative unit price corrected: 18
Negative unit prices remaining: 0


Step 8: Invalid quantities

In [407]:
# Validate quantity values

df['invalid_quantity'] = df['quantity'] <= 0
print('Invalid quantities:', df['invalid_quantity'].sum())

Invalid quantities: 20


Step 9: Validate and correct order totals

In [408]:
#validate order totals

df['calculated_total'] = df['unit_price'] * df['quantity']

In [409]:
# Identify genuine mismatches

df['total_mismatches'] = ~np.isclose(
    df['order_total'],
    df['calculated_total'],
    rtol=1e-09,
    atol=1e-09
)

In [410]:
df['total_difference'] = df['order_total'] - df['calculated_total']

print('Genuine order-total mismatches:', df['total_mismatches'].sum())

Genuine order-total mismatches: 50


In [411]:
df.drop(columns='calculated_total', inplace=True)

In [412]:
df.head()

,order_id,customer_id,customer_details,product_category,unit_price,quantity,order_date,sales_channel,order_total,parsed_customer_name,...,plan,segment,signup_date,_merge,customer_master_match,order_before_signup,negative_price_corrected,invalid_quantity,total_mismatches,total_difference
0,ORD-000001,C00347,Leo Lewis | Germany | Basic,Home & Kitchen,348.61,1,2026-06-10,Website,348.61,Leo Lewis,...,Basic,Small Business,2025-08-19,both,Matched,False,False,False,False,0.0
1,ORD-000002,C00966,Nora Baker | Australia | Plus,Clothing,274.73,5,2025-02-21,Website,1373.65,Nora Baker,...,Plus,Consumer,2025-04-12,both,Matched,True,False,False,False,0.0
2,ORD-000003,C01862,Lina Clark | United Kingdom | Basic,Electronics,194.69,4,2025-02-06,Mobile App,778.76,Lina Clark,...,Basic,Small Business,2023-10-01,both,Matched,False,False,False,False,0.0
3,ORD-000004,C02451,Amelia Hall | France | Basic,Beauty,631.77,5,2025-09-15,Marketplace,3158.85,Amelia Hall,...,Basic,Consumer,2023-08-11,both,Matched,False,False,False,False,0.0
4,ORD-000005,C00730,Adam Walker | Singapore | Plus,Clothing,324.89,1,2025-06-26,Mobile App,324.89,Adam Walker,...,Plus,Consumer,2025-09-04,both,Matched,True,False,False,False,0.0


Step 10: Duplicate Order IDs

In [413]:
# Duplicate order IDs

exact_duplicate_count = df.duplicated().sum()
df['duplicate_order_id'] = df['order_id'].duplicated(keep=False)

print('Exact duplicate rows:', exact_duplicate_count)
print('Rows with duplicate order IDs:', df['duplicate_order_id'].sum())
print('Unique duplicated order IDs:', df.loc[df['duplicate_order_id'], 'order_id'].nunique())

Exact duplicate rows: 0
Rows with duplicate order IDs: 32
Unique duplicated order IDs: 16


Step 11: Missing order dates

In [414]:
df['missing_order_date'] = df['order_date'].isna()
print('missing order dates:', df['missing_order_date'].sum())

missing order dates: 24


Step 12: Consolidated review flag and summary

In [415]:
df['no_customer_match'] = df['customer_master_match'].eq('Not Found')

issue_columns = [
    'missing_order_date',
    'no_customer_match',
    'order_before_signup',
    'negative_price_corrected',
    'invalid_quantity',
    'total_mismatches',
    'duplicate_order_id'
]

issue_mask = df[issue_columns].any(axis=1)
data_quality_exceptions = df.loc[issue_mask].copy()

print('Rows flagged for review:', issue_mask.sum(), f'({issue_mask.mean():.1%})')

Rows flagged for review: 2163 (21.6%)


In [416]:
# Final data-quality summary

summary = pd.DataFrame({
    'check': [
        'Rows',
        'Missing order dates',
        'Customer IDs not found in master',
        'Orders before customer signup',
        'Negative unit prices corrected',
        'Invalid quantities',
        'Order-total mismatches',
        'Rows with duplicate order IDs',
        'Exact duplicate rows'
    ],
    'count': [
        len(df),
        df['order_date'].isna().sum(),
        (df['customer_master_match'] == 'Not Found').sum(),
        df['order_before_signup'].sum(),
        df['negative_price_corrected'].sum(),
        df['invalid_quantity'].sum(),
        df['total_mismatches'].sum(),
        df['duplicate_order_id'].sum(),
        exact_duplicate_count
    ]
})
summary

,check,count
0,Rows,10000
1,Missing order dates,24
2,Customer IDs not found in master,525
3,Orders before customer signup,1545
4,Negative unit prices corrected,18
5,Invalid quantities,20
6,Order-total mismatches,50
7,Rows with duplicate order IDs,32
8,Exact duplicate rows,0


In [417]:
temporary_columns = [
    'parsed_customer_name',
    'parsed_country',
    'parsed_plan',
    '_merge',
]

df = df.drop(
    columns=temporary_columns,
    errors='ignore'
)

In [418]:
missing = [c for c in issue_columns if c not in df.columns]
assert not missing, f"issue_columns references columns that don't exist yet: {missing}"
assert issue_mask.sum() == 2163, f"expected 2163 flagged rows, got {issue_mask.sum()}"

In [419]:
print("Cleaned dataset:", df.shape)
print("Exception rows:", data_quality_exceptions.shape)

print("\nIssue counts:")
print(df[issue_columns].sum())

print("\nAny issue rows:")
print(df[issue_columns].any(axis=1).sum())

Cleaned dataset: (10000, 24)
Exception rows: (2163, 28)

Issue counts:
missing_order_date            24
no_customer_match            525
order_before_signup         1545
negative_price_corrected      18
invalid_quantity              20
total_mismatches              50
duplicate_order_id            32
dtype: int64

Any issue rows:
2163


Step 13: Save Outputs

In [420]:
# save cleaned data

df.to_csv('cleaned_orders.csv', index=False)

data_quality_exceptions.to_csv('data_quality_exceptions.csv', index=False)

In [421]:
print('Cleaned dataset:', df.shape)
print('Exception rows:', data_quality_exceptions.shape[0])

Cleaned dataset: (10000, 24)
Exception rows: 2163
